In [ ]:
import urllib.request
import zipfile
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
%pip install xgboost
%pip install lightgbm
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score
)
from sklearn.preprocessing import LabelEncoder, label_binarize
from tempfile import mkdtemp

import mlflow
import mlflow.sklearn

url = "https://raw.githubusercontent.com/Mafegz0/Data/main/df_morosidad.csv.zip"
zip_path = "/tmp/df_morosidad.zip"

urllib.request.urlretrieve(url, zip_path)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall("/tmp")

df = pd.read_csv("/tmp/df_morosidad.csv")


TARGET = "y_categorica"
DROP = ["Llave2", "Nombre_linea", "IDBANNER"]

y = df[TARGET].copy()
X = df.drop([TARGET] + [c for c in DROP if c in df.columns], axis=1).copy()

num_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

le = LabelEncoder().fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)
n_clases = len(le.classes_)

prep = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols),
    ],
    remainder="drop"
)


modelos = {
    "RandomForest": RandomForestClassifier(
        bootstrap=True,
        max_depth=22,
        max_features=0.5,
        min_samples_leaf=5,
        n_estimators=479,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        objective="multi:softprob",
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        n_estimators=244,
        max_depth=5,
        learning_rate=0.0490198317,
        subsample=0.71825347433,
        colsample_bytree=0.63079196393,
        gamma=0.14487572645,
        min_child_weight=2
    ),

    "LightGBM": LGBMClassifier(
        objective="multiclass",
        num_class=n_clases,
        metric="multi_logloss",
        n_jobs=-1,
        random_state=42,
        n_estimators=861,
        learning_rate=0.0261899338,
        num_leaves=54,
        max_depth=3,
        min_child_samples=45,
        subsample=0.6831766651472755,
        colsample_bytree=0.798070764044508,
        reg_alpha=0.37768070515882624,
        reg_lambda=0.21257793724562235
    )
}

resultados = []

for nombre, modelo in modelos.items():

    pipe = Pipeline([
        ("prep", prep),
        ("clf", modelo)
    ])
    pipe.set_params(memory=mkdtemp())

    experiment = mlflow.set_experiment(f"/modelo-{nombre.lower()}-morosidad")

    with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=nombre):

        pipe.fit(X_train, y_train_enc)

        proba_test = pipe.predict_proba(X_test)
        y_test_bin = label_binarize(y_test_enc, classes=range(n_clases))

        test_auc = roc_auc_score(
            y_test_bin, proba_test,
            multi_class="ovr", average="macro"
        )

        y_pred_enc = pipe.predict(X_test)
        y_pred = le.inverse_transform(y_pred_enc)

        acc  = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
        rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)

        # Log parámetros
        for p, v in modelo.get_params().items():
            mlflow.log_param(p, v)

        # Log métricas
        mlflow.log_metric("AUC_macro_OVR", float(test_auc))
        mlflow.log_metric("Accuracy", float(acc))
        mlflow.log_metric("Precision_macro", float(prec))
        mlflow.log_metric("Recall_macro", float(rec))

        mlflow.sklearn.log_model(pipe, f"modelo_{nombre.lower()}")

        resultados.append([nombre, test_auc, acc, prec, rec])



df_resultados = pd.DataFrame(
    resultados,
    columns=["Modelo", "AUC", "Accuracy", "Precision_macro", "Recall_macro"]
)

print("\n\n=== RESULTADOS COMPARATIVOS ===\n")
display(df_resultados.sort_values(by="AUC", ascending=False))